In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Create the same mock dataset
mock_articles = [
    {
        'text': "The government's new policy on farmers has received widespread support across all political parties.",
        'label': 0,  # neutral
        'bias_type': 'political'
    },
    {
        'text': "Opposition parties continue their baseless criticism of the visionary leadership's groundbreaking reforms.",
        'label': 1,  # pro-government
        'bias_type': 'political'
    },
    {
        'text': "The authoritarian regime's latest draconian measures have sparked outrage among civil rights activists.",
        'label': 2,  # anti-government
        'bias_type': 'political'
    },
    {
        'text': "The festival celebrations brought together people from all communities in unity.",
        'label': 0,  # neutral
        'bias_type': 'communal'
    },
    {
        'text': "The minority community's demands are creating unnecessary tension in our peaceful society.",
        'label': 3,  # communal bias
        'bias_type': 'communal'
    },
    {
        'text': "South Indian states consistently outperform others in education and development metrics.",
        'label': 4,  # regional bias
        'bias_type': 'regional'
    },
    {
        'text': "The reservation system benefits undeserving candidates while meritorious students face discrimination.",
        'label': 5,  # caste bias
        'bias_type': 'caste'
    },
    {
        'text': "The Supreme Court's landmark judgment ensures equal rights for all citizens.",
        'label': 0,  # neutral
        'bias_type': 'legal'
    }
]

# Generate more mock data
additional_texts = [
    ("The parliamentary session discussed various bills with constructive debate.", 0),
    ("The dynamic leadership's innovative policies have revolutionized India's standing.", 1),
    ("The government's failures have led to widespread public discontent.", 2),
    ("Traditional values are under attack from foreign-influenced ideologies.", 3),
    ("Western India drives the nation's progress while other regions lag behind.", 4),
    ("Economic indicators show mixed trends across different sectors.", 0),
    ("International experts praise the government's swift action on reforms.", 1),
    ("Opposition leaders expose the administration's broken promises.", 2),
    ("The secular facade cannot hide preferential treatment to certain communities.", 3),
    ("Northern states need to learn from progressive southern regions.", 4)
]

for text, label in additional_texts:
    mock_articles.append({
        'text': text,
        'label': label,
        'bias_type': 'generated'
    })

df = pd.DataFrame(mock_articles)
print(f"Created dataset with {len(df)} articles")
print(df['label'].value_counts())

In [ ]:
def test_pretrained_models_simple():
    """Test pre-trained models without complex training"""
    
    # Models that should work
    working_models = [
        "distilbert-base-uncased-finetuned-sst-2-english",
        "cardiffnlp/twitter-roberta-base-sentiment-latest"
    ]
    
    results = {}
    
    for model_name in working_models:
        try:
            print(f"\n--- Testing {model_name} ---")
            classifier = pipeline("text-classification", model=model_name)
            
            # Test on sample articles
            sample_texts = df['text'].head(5).tolist()
            predictions = classifier(sample_texts)
            
            results[model_name] = predictions
            
            for i, (text, pred) in enumerate(zip(sample_texts, predictions)):
                actual_label = df.iloc[i]['label']
                print(f"Article {i+1}:")
                print(f"  Text: {text[:80]}...")
                print(f"  Predicted: {pred['label']} (score: {pred['score']:.3f})")
                print(f"  Actual bias type: {df.iloc[i]['bias_type']}")
                print()
                
        except Exception as e:
            print(f"Error with {model_name}: {str(e)}")
    
    return results

# Test pre-trained models
print("=== Testing Pre-trained Models ===")
pretrained_results = test_pretrained_models_simple()

In [ ]:
class BiasDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

def train_simple_model():
    """Simple training without Trainer class"""
    
    # Prepare data
    texts = df['text'].tolist()
    labels = df['label'].tolist()
    
    # Split data
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        texts, labels, test_size=0.2, random_state=42
    )
    
    print(f"Training samples: {len(train_texts)}")
    print(f"Validation samples: {len(val_texts)}")
    
    # Initialize tokenizer and model
    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    num_labels = len(set(labels))
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, 
        num_labels=num_labels
    )
    
    # Create datasets
    train_dataset = BiasDataset(train_texts, train_labels, tokenizer)
    val_dataset = BiasDataset(val_texts, val_labels, tokenizer)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
    
    # Training setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss()
    
    # Training loop
    num_epochs = 2  # Reduced for quick testing
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        
        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            if batch_idx % 2 == 0:  # Print every 2 batches
                print(f"  Batch {batch_idx}, Loss: {loss.item():.4f}")
        
        avg_loss = total_loss / len(train_loader)
        print(f"  Average Loss: {avg_loss:.4f}")
        
        # Validation
        model.eval()
        val_predictions = []
        val_true_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                predictions = torch.argmax(outputs.logits, dim=1)
                
                val_predictions.extend(predictions.cpu().numpy())
                val_true_labels.extend(labels.cpu().numpy())
        
        val_accuracy = accuracy_score(val_true_labels, val_predictions)
        print(f"  Validation Accuracy: {val_accuracy:.4f}")
    
    # Save model
    model.save_pretrained('./simple_bias_model')
    tokenizer.save_pretrained('./simple_bias_model')
    
    print("Model saved successfully!")
    return model, tokenizer

# Train simple model
print("\n=== Training Simple Model ===")
try:
    trained_model, trained_tokenizer = train_simple_model()
    print("Training completed successfully!")
except Exception as e:
    print(f"Training error: {str(e)}")
    print("Continuing with pre-trained models only...")

In [ ]:
def create_simple_classifier():
    """Create a simple classifier using available models"""
    
    # Try to load custom model first, fallback to pre-trained
    try:
        classifier = pipeline("text-classification", model="./simple_bias_model")
        print("Using custom trained model")
    except:
        # Fallback to pre-trained
        classifier = pipeline("text-classification", 
                            model="distilbert-base-uncased-finetuned-sst-2-english")
        print("Using pre-trained model")
    
    return classifier

def analyze_articles(classifier):
    """Analyze articles for bias"""
    
    test_articles = [
        "The Prime Minister's latest initiative has transformed India's infrastructure.",
        "Opposition parties raise concerns about the government's economic policies.",
        "The festival brought together people from all religions in celebration.",
        "Some communities are getting unfair advantages in government schemes.",
        "The northeastern states have unique cultural heritage."
    ]
    
    print("\n=== Article Analysis Results ===")
    
    for i, article in enumerate(test_articles, 1):
        try:
            result = classifier(article)
            print(f"\nArticle {i}:")
            print(f"Text: {article}")
            print(f"Classification: {result[0]['label']}")
            print(f"Confidence: {result[0]['score']:.3f}")
            
            # Simple bias interpretation
            score = result[0]['score']
            if score > 0.8:
                bias_level = "High confidence"
            elif score > 0.6:
                bias_level = "Medium confidence"
            else:
                bias_level = "Low confidence"
            
            print(f"Bias Detection: {bias_level}")
            
        except Exception as e:
            print(f"Error analyzing article {i}: {str(e)}")

# Create and test classifier
classifier = create_simple_classifier()
analyze_articles(classifier)

print("\n=== Summary ===")
print("✅ Mock dataset created")
print("✅ Pre-trained models tested")
print("✅ Simple training approach implemented")
print("✅ Inference pipeline ready")
print("\nTo fix the original error, run: pip install accelerate>=0.26.0")

In [ ]:
def create_ensemble_bias_detector():
    """Create ensemble model combining multiple approaches"""
    
    class EnsembleBiasDetector:
        def __init__(self):
            self.models = {}
            self.load_models()
        
        def load_models(self):
            """Load multiple models for ensemble"""
            try:
                # Load custom model
                self.models['custom'] = pipeline(
                    "text-classification", 
                    model="./indian_news_bias_model_final"
                )
                print("Loaded custom model")
            except:
                print("Custom model not available")
            
            # Try to load other models
            backup_models = [
                "cardiffnlp/twitter-roberta-base-sentiment-latest",
                "distilbert-base-uncased-finetuned-sst-2-english"
            ]
            
            for model_name in backup_models:
                try:
                    self.models[model_name] = pipeline("text-classification", model=model_name)
                    print(f"Loaded {model_name}")
                    break
                except:
                    continue
        
        def predict(self, text):
            """Get ensemble prediction"""
            results = {}
            
            for name, model in self.models.items():
                try:
                    prediction = model(text)
                    results[name] = prediction[0]
                except Exception as e:
                    results[name] = {'error': str(e)}
            
            return results
        
        def analyze_article(self, article):
            """Comprehensive article analysis"""
            predictions = self.predict(article)
            
            analysis = {
                'article': article[:200] + "..." if len(article) > 200 else article,
                'predictions': predictions,
                'summary': self.summarize_predictions(predictions)
            }
            
            return analysis
        
        def summarize_predictions(self, predictions):
            """Summarize ensemble predictions"""
            summary = {"bias_detected": False, "confidence": 0.0, "consensus": "unknown"}
            
            valid_predictions = [p for p in predictions.values() if 'error' not in p]
            
            if valid_predictions:
                # Simple majority voting (can be improved)
                labels = [p.get('label', 'UNKNOWN') for p in valid_predictions]
                scores = [p.get('score', 0.0) for p in valid_predictions]
                
                if scores:
                    summary['confidence'] = np.mean(scores)
                    summary['consensus'] = max(set(labels), key=labels.count) if labels else "unknown"
                    summary['bias_detected'] = summary['confidence'] > 0.7
            
            return summary

    return EnsembleBiasDetector()

# Create and test ensemble
ensemble_detector = create_ensemble_bias_detector()

# Test with sample articles
test_cases = [
    "The government's new policy shows their commitment to inclusive development.",
    "Religious minorities are facing increasing persecution in recent times.",
    "The court delivered a fair judgment considering all aspects of the case."
]

print("\n=== Ensemble Analysis Results ===")
for i, article in enumerate(test_cases, 1):
    print(f"\n--- Test Case {i} ---")
    analysis = ensemble_detector.analyze_article(article)
    print(f"Article: {analysis['article']}")
    print(f"Summary: {analysis['summary']}")
    for model, result in analysis['predictions'].items():
        if 'error' not in result:
            print(f"{model}: {result.get('label', 'N/A')} ({result.get('score', 0):.3f})")

In [ ]:
def evaluate_bias_detection():
    """Comprehensive evaluation of bias detection models"""
    
    # Create evaluation dataset
    eval_data = [
        {"text": "The policy benefits all sections of society equally.", "true_label": "neutral"},
        {"text": "The great leader's visionary policies will make India a superpower.", "true_label": "pro_government"},
        {"text": "The corrupt government continues to betray public trust.", "true_label": "anti_government"},
        {"text": "Some communities always create problems for peaceful coexistence.", "true_label": "communal_bias"},
        {"text": "Northern states are less developed compared to southern counterparts.", "true_label": "regional_bias"}
    ]
    
    print("=== Model Evaluation Results ===")
    
    # Test ensemble detector if available
    if 'ensemble_detector' in globals():
        correct = 0
        total = len(eval_data)
        
        for item in eval_data:
            analysis = ensemble_detector.analyze_article(item['text'])
            predicted = analysis['summary']['consensus'].lower()
            actual = item['true_label'].lower()
            
            is_correct = predicted in actual or actual in predicted
            if is_correct:
                correct += 1
                
            print(f"Text: {item['text'][:60]}...")
            print(f"Actual: {actual}, Predicted: {predicted}, Correct: {is_correct}")
            print()
        
        accuracy = correct / total
        print(f"Overall Accuracy: {accuracy:.2%}")

# Run evaluation
evaluate_bias_detection()

print("\n=== Setup Complete ===")
print("You now have:")
print("1. Mock Indian news dataset")
print("2. Pre-trained model testing")
print("3. Custom fine-tuned model")
print("4. Ensemble approach")
print("5. Evaluation framework")